# Behavioral Alignment: Multi-Model Comparison

This notebook demonstrates IRTK's `behavior_alignment` module for comparing
mechanistic solutions across models:

1. **Mechanical correspondence** – matching heads across models
2. **Solution diversity** – how differently models solve the same task
3. **Alignment spectrum** – layer-by-layer similarity between models
4. **Interpretability transfer** – do interventions from one model work in another?
5. **Emergence comparison** – when features appear across model scales

## Why This Matters

Multi-model comparison reveals whether different architectures learn the same algorithms. Head correspondence, circuit alignment, and interpretability transfer tests address the universality question: are the circuits we find in GPT-2 also present in other models?

**Key references:**
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.behavior_alignment import (
    mechanical_correspondence,
    solution_diversity,
    behavioral_alignment_spectrum,
    interpretability_transfer,
    emergence_comparison,
)

In [ ]:
# Build two small models with different random seeds
def make_model(seed=42):
    cfg = HookedTransformerConfig(
        n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50,
    )
    model = HookedTransformer(cfg)
    key = jax.random.PRNGKey(seed)
    leaves, treedef = jax.tree.flatten(model)
    new_leaves = []
    for leaf in leaves:
        if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
            key, subkey = jax.random.split(key)
            new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
        else:
            new_leaves.append(leaf)
    return jax.tree.unflatten(treedef, new_leaves)

model_a = make_model(seed=42)
model_b = make_model(seed=99)
seqs = [jnp.array([0, 1, 2, 3, 4, 5]), jnp.array([10, 11, 12, 13])]
print("Models built: model_a (seed=42), model_b (seed=99)")

## 1. Mechanical Correspondence

Find which attention heads in model A correspond to heads in model B by
comparing their attention patterns.

In [ ]:
corr = mechanical_correspondence(model_a, model_b, seqs)

print(f"Correspondence matrix shape: {corr['head_correspondence'].shape}")
print(f"Mean correspondence: {corr['mean_correspondence']:.4f}")
print(f"Residual similarity: {corr['residual_similarity']:.4f}")
print(f"\nBest matches (head_a -> head_b, score):")
for ha, hb, score in corr["best_matches"]:
    print(f"  Head {ha} -> Head {hb}: {score:.4f}")

## 2. Solution Diversity

Measure how differently the two models solve the same task — do they agree
on top-1 predictions?

In [ ]:
def metric_fn(logits):
    return float(logits[-1, 0])

div = solution_diversity([model_a, model_b], seqs, metric_fn)

print(f"Metric values shape: {div['metric_values'].shape}")
print(f"Metric variance: {div['metric_variance']:.4f}")
print(f"Logit agreement: {div['logit_agreement']:.4f}")
print(f"Diversity score: {div['diversity_score']:.4f}")
print(f"  (0 = identical, 1 = maximally diverse)")

## 3. Behavioral Alignment Spectrum

Track how aligned two models are from input through each layer to output.

In [ ]:
spectrum = behavioral_alignment_spectrum(model_a, model_b, seqs)

print(f"Layer similarities: {spectrum['layer_similarities'].round(4)}")
print(f"Output similarity: {spectrum['output_similarity']:.4f}")
print(f"Divergence layer: {spectrum['divergence_layer']}")
print(f"Alignment AUC: {spectrum['alignment_auc']:.4f}")

## 4. Interpretability Transfer

Test whether an intervention (activation patch) discovered in the source
model transfers to the target model.

In [ ]:
transfer = interpretability_transfer(
    model_a, model_b, seqs, "blocks.0.hook_resid_post", metric_fn
)

print(f"Source metrics: {transfer['source_metrics'].round(4)}")
print(f"Target baseline: {transfer['target_baseline'].round(4)}")
print(f"Target transferred: {transfer['target_transferred'].round(4)}")
print(f"Transfer success rate: {transfer['transfer_success']:.1%}")
print(f"Mean transfer effect: {transfer['mean_transfer_effect']:.4f}")

## 5. Emergence Comparison

Compare when features emerge at a hook point across different models
(simulating different scales or training checkpoints).

In [ ]:
emergence = emergence_comparison(
    [model_a, model_b], seqs, "blocks.0.hook_resid_post"
)

print(f"Activation norms shape: {emergence['activation_norms'].shape}")
print(f"Sparsity levels: {emergence['sparsity_levels'].round(4)}")
print(f"Feature overlap:\n{emergence['feature_overlap'].round(4)}")
print(f"Emergence order (strongest model): {emergence['emergence_order']}")